# RLHF
<div align="center">

<table border="1" style="margin: 0 auto;">
<tr>
  <td style="text-align: center;"><strong>对比维度</strong></td>
  <td style="text-align: center;"><strong>监督微调 (SFT)</strong></td>
  <td style="text-align: center;"><strong>人类反馈强化学习 (RLHF)</strong></td>
</tr>
<tr>
  <td style="text-align: center;"><strong>核心目标</strong></td>
  <td style="text-align: center;">模仿正确答案（指令遵循）</td>
  <td style="text-align: center;">对齐人类偏好（有用/无害/诚实）</td>
</tr>
<tr>
  <td style="text-align: center;"><strong>数据需求</strong></td>
  <td style="text-align: center;">高质量的 <code>(指令, 回答)</code> 对</td>
  <td style="text-align: center;">提示词 <code>(Prompt)</code> + 偏好排序/评分</td>
</tr>
<tr>
  <td style="text-align: center;"><strong>学习方式</strong></td>
  <td style="text-align: center;">填鸭式教学（拟合分布）</td>
  <td style="text-align: center;">探索式学习（试错与反馈）</td>
</tr>
<tr>
  <td style="text-align: center;"><strong>优化信号</strong></td>
  <td style="text-align: center;">Token 级预测概率（交叉熵）</td>
  <td style="text-align: center;">整句生成质量评分（标量奖励）</td>
</tr>
<tr>
  <td style="text-align: center;"><strong>泛化能力</strong></td>
  <td style="text-align: center;">局限于训练数据分布</td>
  <td style="text-align: center;">可泛化到未见过的复杂指令</td>
</tr>
</table>

<p><em> SFT 与 RLHF 核心差异对比</em></p>

</div>
在 RLHF 框架中，将 Token 生成过程建模为 **“片段的（Episodic）马尔可夫决策过程（MDP）”** 是连接 NLP 与 RL 的核心桥梁。这个“片段”特指“从接收用户提示到生成完整输出”的单次生成任务。

- **状态 (State, $s_t$)**：当前已生成的上下文，包括用户提示 $x$ 和已生成的 Token 序列 $y_{<t}$。即 $s_t = (x, y_1, ..., y_{t-1})$。
- **动作 (Action, $a_t$)**：模型在当前时间步生成的下一个 Token $y_t$。
- **策略 (Policy, $\pi_\theta$)**：即我们的大语言模型。$\pi(a_t|s_t)$ 对应模型在当前上下文下预测下一个 Token 的概率分布。
- **奖励 (Reward, $R$)**：通常在整个回答生成完毕（Episode 结束）后，由奖励模型给予一个标量反馈 $R(x, y)$。中间步骤的奖励通常为 0。

在这个定义下，RLHF 的目标就是寻找一个最优策略 $\pi_\theta$，使得生成的完整序列期望奖励最大化：

$$
\max_\pi \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi(\cdot|x)} [R(x, y)]
$$
首先通过有监督微调得到初始策略模型；然后，收集人类偏好数据训练一个奖励模型；最后，使用奖励模型作为信号，通过强化学习算法（如 PPO）进一步优化策略模型。<br>
（1）收集人类偏好数据

第一步需要准备一个指令（prompt），可从真实用户请求中选取，也可由标注员设计；让 SFT 模型针对该指令生成多个（InstructGPT 中为 4-9 个）不同的回答；由人工标注员对这些回答进行**排序（Ranking）**，从最好到最差。相比于给每个回答打一个绝对分数，排序是一种对人类更友好、也更一致的标注方式。这些排序数据构成了偏好数据集 $\mathcal{D} = \{(x, y_w, y_l)\}_{i=1}^N$，其中 $y_w$ 是比 $y_l$ 更受偏好的回答 (winner)，$y_l$ 是较差的回答 (loser)。

（2）训练奖励模型

训练奖励模型的常见做法包括以下几点。

- 将一个包含 K 个回答的排序数据，转换成 $\binom{K}{2}$ 个成对的比较数据。例如，(回答A > 回答B)、(回答A > 回答C)、(回答B > 回答C)...

- 奖励模型（RM）通常和我们正在优化的语言模型结构类似（但可以小得多），它的任务不是生成文本，而是**输入 `(prompt, response)`，输出一个标量分数 `reward`**。

- 训练 RM 的目标是，让它给人类偏好的回答打出更高的分数。这通常基于 **Bradley-Terry 模型**，该模型假设人类对两个回答的偏好概率可以用它們的潜在奖励分数通过 Sigmoid 函数来建模：

  $$
  P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))
  $$
    
  其损失函数（负对数似然）如下：

  $$
  \text{loss}(\theta) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log(\sigma(r_\theta(x, y_w) - r_\theta(x, y_l))) \right]
  $$

  其中 $r_\theta$ 是奖励模型，$y_w$ 是比 $y_l$ 更受偏好的回答，$\sigma$ 是 Sigmoid 函数。这个损失函数旨在最大化偏好回答与非偏好回答之间的分数差距。

# PPO
## 近端优化策略
在尝试最大化奖励的同时，通过一个约束项来限制新旧策略的差异范围，继而避免单步更新过大导致训练崩溃。它通过优化一个“替代目标函数”（Surrogate Objective）来实现这一点。PPO 最常用的替代目标是 **Clipped Surrogate Objective**:

$$
L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right) \right]
$$

其中，$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ 是新旧策略的概率比，$\hat{A}_t$ 是优势函数估计，$\epsilon$ 是一个限制更新范围的超参数（通常为 0.2）。这个裁剪操作有效地构建了一个悲观的下界，防止策略更新过于激进。</br>
单纯用 PPO 优化奖励模型，可能会导致模型在某些传统 NLP 任务（如 SQuAD）上的性能下降，这种现象被称为 **“对齐税”（Alignment Tax）**。为了解决这个问题，InstructGPT 提出了一种名为 **PPO-ptx** 的变体，它在优化奖励的同时，混合了一部分预训练数据的梯度，其优化目标为：

$$
\text{objective}(\phi) = \mathbb{E}_{(x, y) \sim D_{\pi_\phi^{RL}}} [r_\theta(x, y) - \beta \log(\pi_\phi^{RL}(y|x) / \pi^{SFT}(y|x))] + \gamma \mathbb{E}_{x \sim D_{pretrain}}[\sum_t \log(\pi_\phi^{RL}(x_t|x_{<t}))]
$$

其中：
- PPO 主目标项（奖励 + KL 约束）：在当前策略生成的数据分布 $D_{\pi_\phi^{RL}}$ 上，最大化奖励模型分数 $r_\theta(x,y)$，并用每 token KL 约束惩罚新策略 $\pi_\phi^{RL}$ 偏离参考模型 $\pi^{SFT}$，以抑制灾难性漂移。

- 预训练保留项（ptx）：在预训练语料 $D_{pretrain}$ 上进行最大似然学习 $\sum_t \log \pi_\phi^{RL}(x_t|x_{<t})$，用于保留通用语言能力与知识面，缓解“对齐税”。

- 符号说明：$\beta$ 控制 KL 约束强度；$\gamma$ 控制 ptx 项权重；$D_{\pi_\phi^{RL}}$ 为在线由当前策略采样的数据分布；$D_{pretrain}$ 为静态预训练语料；KL 为每 token KL 散度。

# DPO
## 直接偏好优化
$$ \mathcal{L}{\text{DPO}}(\pi\theta; \pi_{\text{ref}}) = - \mathbb{E}{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right] $$

其中 
π
ref
 是参考模型（通常是 SFT 模型），$\sigma$ 是 Sigmoid 函数。这个损失函数简洁地鼓励模型提高 
y
w
 的概率，降低 
y
l
 的概率。
 DPO 的梯度更新有一个直观的解释。它会给那些被隐式奖励模型错误排序的样本（即认为 
y
l
 比 
y
w
 更好）更大的权重，从而集中火力修正模型的“判断失误”。这个隐式奖励由下式定义： $$ \hat r_\theta(x,y)=\beta \log \frac{\pi_\theta(y|x)}{\pi_{\mathrm{ref}}(y|x)} $$

# GRPO
## 群体相对策略优化
**RLVR 的核心逻辑**是用确定性的验证器（Validator）取代主观的奖励模型（如通过代码测试用例 $v(x, y) \in \{0, 1\}$ 直接判断正误）。这种强反馈消除了 Reward Hacking 中“欺骗人类”的可能性，并驱动模型通过**自我博弈（Self-Play）**涌现出思维链（CoT）等高级推理能力。然而，传统的 PPO 算法需要维护一个与策略模型规模相当的价值网络（Critic），在训练千亿参数模型时显存开销巨大。GRPO 的关键创新是**“去 Critic 化”**，它通过组内统计量来替代 Critic 的基线估计功能：

- **组采样 (Group Sampling)**：对于同一个提示词 $x$，利用旧策略 $\pi_{\theta_{old}}$ 采样生成一组输出 $\{y_1, y_2, ..., y_G\}$。

- **组内优势估计 (Group Relative Advantage)**：利用奖励模型或验证器对这组输出打分。关键在于，GRPO 不使用额外的 Critic 来预测基线，而是直接使用**组内奖励的均值**作为动态基线。这种方法不仅显著降低了显存占用（无需维护与策略模型同等规模的 Critic），还有效利用了组内对比来降低方差。对于每个输出，其优势 $A_i$ 通常通过标准化计算得出：

  $$
  A_i = \frac{r_i - \text{mean}(\{r_1, ..., r_G\})}{\text{std}(\{r_1, ..., r_G\})}
  $$

- **KL 散度正则化**：与 PPO 通常在奖励函数中直接扣除 KL 惩罚不同，GRPO 将 KL 散度 $D_{KL}(\pi_\theta || \pi_{ref})$ 直接加入损失函数中作为正则项。这样做的好处是保持了优势估计的纯粹性，使其仅反映组内输出的相对好坏，而不会被 KL 惩罚项混淆。